# MODFLOW 6 first pass (synthetic test model)

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Tests that MODFLOW 6 + FloPy run in Colab. **This is not yet a model of Alwar.** It uses a synthetic 20 × 20 grid of 1 km cells that is not georeferenced, one 50 m layer (assumed), uniform K = 6.89 m/d (from the midpoint of the CGWB transmissivity range 27–662 m²/d), Sy = 0.10 (assumed), constant heads on the west and east edges, and uniform recharge. The calibration-preparation cell found **0 numeric water levels** because it did not recognise the `WL_bgl` column and the file has no coordinates.

| | |
|---|---|
| **Reads** | CGWB transmissivity range; `groundwater_cleaned_long_seasonal.csv` |
| **Writes** | `/content/Alwar_MODFLOW6_FirstPass` (temporary; lost when Colab restarts) → ZIP copied to `gw_framework/outputs/`; `MyDrive/Alwar_Groundwater_Calibration/` |
| **Cells** | 4 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. Install MODFLOW 6
- 2. First-pass model
- 3. Calibration preparation
- 4. Save to Drive

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["flopy"], modflow=False)
del TOKEN, URL


## 1. Install MODFLOW 6

**[orig #30]** Install MODFLOW 6 in Colab  
<sub>✅ ran OK</sub>

In [ ]:

# Fix MODFLOW 6 installation in Google Colab

import os
import sys
import subprocess
from pathlib import Path

import flopy

bindir = Path("/content/modflow6_bin")
bindir.mkdir(parents=True, exist_ok=True)

# Install MODFLOW 6 using the utility available through FloPy.
flopy.utils.get_modflow(
    str(bindir),
    repo="modflow6"
)

# Locate the executable.
candidates = list(bindir.rglob("mf6"))
if not candidates:
    candidates = list(bindir.rglob("mf6.exe"))

if not candidates:
    raise FileNotFoundError(
        f"MODFLOW 6 executable was not found in {bindir}"
    )

mf6_exe = str(candidates[0])
os.chmod(mf6_exe, 0o755)

# Verify the executable.
check = subprocess.run(
    [mf6_exe],
    capture_output=True,
    text=True,
    timeout=30
)

print("Executable:", mf6_exe)
print("Return code:", check.returncode)
print((check.stdout or check.stderr)[:1500])

# Keep this variable for the model-building cell.
print("\nInstallation check completed.")


## 2. First-pass model

**[orig #31]** First-pass MODFLOW 6 model (20 × 20 synthetic grid, 1 layer, 12 periods)  
<sub>✅ ran OK</sub>

In [ ]:

# =====================================================================
# ALWAR GROUNDWATER MODEL — COMPLETE GOOGLE COLAB SCRIPT
# MODFLOW 6 + FloPy 3.11+
#
# Model: single-layer, equivalent porous medium
# Purpose: first numerical prototype and sensitivity preparation
#
# NOT YET A CALIBRATED DISTRICT MODEL.
# Provisional values are explicitly documented below.
# =====================================================================

import os
import sys
import json
import shutil
import subprocess
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# 1. INSTALL / IMPORT FLOPY
# ---------------------------------------------------------------------

try:
    import flopy
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "flopy"
    ])
    import flopy

print("FloPy version:", flopy.__version__)

# MODFLOW 6 was installed successfully in your previous cell.
# Use the exact executable path confirmed by your Colab log.
MF6_EXE = Path("/content/modflow6_bin/mf6")

# If the current Colab runtime was restarted, reinstall MODFLOW 6.
if not MF6_EXE.exists():
    print("MODFLOW 6 executable not found. Installing it through FloPy...")

    MF6_DIR = Path("/content/modflow6_bin")
    MF6_DIR.mkdir(parents=True, exist_ok=True)

    flopy.utils.get_modflow(
        str(MF6_DIR),
        repo="modflow6"
    )

    if not MF6_EXE.exists():
        candidates = list(MF6_DIR.rglob("mf6"))
        if not candidates:
            raise FileNotFoundError(
                "MODFLOW 6 executable was not found after installation."
            )
        MF6_EXE = candidates[0]

os.chmod(MF6_EXE, 0o755)
print("MODFLOW 6 executable:", MF6_EXE)

# Do not run mf6 alone here. It expects a simulation name file.
# FloPy will invoke it inside the correct simulation workspace.

# ---------------------------------------------------------------------
# 2. OUTPUT DIRECTORIES
# ---------------------------------------------------------------------

OUT = Path("/content/Alwar_MODFLOW6_FirstPass")
SIM_WS = OUT / "simulation"

OUT.mkdir(parents=True, exist_ok=True)
SIM_WS.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# 3. MODEL DOMAIN AND DISCRETIZATION
# ---------------------------------------------------------------------
# These are numerical placeholders, not an actual Alwar district GIS grid.
# Replace them after defining a verified model boundary.

nlay = 1
nrow = 20
ncol = 20

delr = 1000.0          # Cell width, m
delc = 1000.0          # Cell height, m

top_elevation = 100.0  # Illustrative model datum, m
bottom_elevation = 50.0
layer_thickness = top_elevation - bottom_elevation

# Initial heads must lie within the model's vertical range.
initial_head = 97.5

# ---------------------------------------------------------------------
# 4. HYDRAULIC PARAMETERS AND EVIDENCE STATUS
# ---------------------------------------------------------------------
# CGWB Alwar report:
# Transmissivity T = 27 to 662 m2/day
# Storativity S = 5e-5 to 2.6e-2
#
# T is NOT hydraulic conductivity.
# K = T / b is used only as an illustrative conversion because
# the 50 m model thickness has not been verified as aquifer thickness.

T_min = 27.0
T_max = 662.0
T_initial = (T_min + T_max) / 2.0

K_min = T_min / layer_thickness
K_max = T_max / layer_thickness
K_initial = T_initial / layer_thickness

# Provisional storage parameters: not calibrated or measured locally.
Sy_initial = 0.10
Ss_initial = 1.0e-5  # 1/m

# Provisional uniform recharge; NOT a measured or calibrated Alwar rate.
recharge_mm_year = 100.0
recharge_m_day = recharge_mm_year / 1000.0 / 365.25

# Illustrative boundary heads, not observed Alwar heads.
west_head = 100.0
east_head = 95.0

# Twelve equal 30-day periods = 360-day model year.
# For research, use actual dates and suitable period lengths.
nper = 12
period_days = 30.0

# ---------------------------------------------------------------------
# 5. WRITE A PARAMETER REGISTER
# ---------------------------------------------------------------------

parameters = [
    {
        "parameter": "Transmissivity_min",
        "value": T_min,
        "units": "m2/day",
        "basis": "CGWB Alwar report",
        "status": "Reported range"
    },
    {
        "parameter": "Transmissivity_max",
        "value": T_max,
        "units": "m2/day",
        "basis": "CGWB Alwar report",
        "status": "Reported range"
    },
    {
        "parameter": "Transmissivity_initial",
        "value": T_initial,
        "units": "m2/day",
        "basis": "Midpoint of reported range",
        "status": "Initial scenario only"
    },
    {
        "parameter": "Layer_thickness",
        "value": layer_thickness,
        "units": "m",
        "basis": "Illustrative model geometry",
        "status": "Assumed, not mapped"
    },
    {
        "parameter": "K_min",
        "value": K_min,
        "units": "m/day",
        "basis": "T_min / assumed layer thickness",
        "status": "Derived scenario bound, not measured"
    },
    {
        "parameter": "K_initial",
        "value": K_initial,
        "units": "m/day",
        "basis": "T_initial / assumed layer thickness",
        "status": "Derived initial value, not measured"
    },
    {
        "parameter": "K_max",
        "value": K_max,
        "units": "m/day",
        "basis": "T_max / assumed layer thickness",
        "status": "Derived scenario bound, not measured"
    },
    {
        "parameter": "Specific_yield",
        "value": Sy_initial,
        "units": "dimensionless",
        "basis": "No verified local value assigned",
        "status": "Provisional sensitivity value"
    },
    {
        "parameter": "Specific_storage",
        "value": Ss_initial,
        "units": "1/m",
        "basis": "No verified local value assigned",
        "status": "Provisional sensitivity value"
    },
    {
        "parameter": "Recharge",
        "value": recharge_mm_year,
        "units": "mm/year",
        "basis": "No calibrated recharge rate assigned",
        "status": "Provisional sensitivity value"
    },
    {
        "parameter": "West_boundary_head",
        "value": west_head,
        "units": "m illustrative datum",
        "basis": "Numerical setup",
        "status": "Assumed boundary condition"
    },
    {
        "parameter": "East_boundary_head",
        "value": east_head,
        "units": "m illustrative datum",
        "basis": "Numerical setup",
        "status": "Assumed boundary condition"
    }
]

parameter_df = pd.DataFrame(parameters)
parameter_df.to_csv(OUT / "parameter_register.csv", index=False)

print("\nParameter register:")
print(parameter_df.to_string(index=False))

# ---------------------------------------------------------------------
# 6. BUILD MODFLOW 6 SIMULATION
# ---------------------------------------------------------------------

sim = flopy.mf6.MFSimulation(
    sim_name="alwar_firstpass",
    sim_ws=str(SIM_WS),
    exe_name=str(MF6_EXE)
)

# Simulation timing
tdis = flopy.mf6.ModflowTdis(
    sim,
    time_units="DAYS",
    nper=nper,
    perioddata=[
        (period_days, 1, 1.0)
        for _ in range(nper)
    ]
)

# Solver
ims = flopy.mf6.ModflowIms(
    sim,
    print_option="SUMMARY",
    complexity="MODERATE",
    outer_dvclose=1.0e-4,
    outer_maximum=200,
    inner_maximum=300,
    inner_dvclose=1.0e-5,
    linear_acceleration="BICGSTAB"
)

# Groundwater flow model
gwf = flopy.mf6.ModflowGwf(
    sim,
    modelname="alwar_gwf",
    save_flows=True
)

sim.register_ims_package(ims, [gwf.name])

# ---------------------------------------------------------------------
# 7. GRID AND INITIAL HEADS
# ---------------------------------------------------------------------

top = np.full((nrow, ncol), top_elevation, dtype=float)
botm = np.full(
    (nlay, nrow, ncol),
    bottom_elevation,
    dtype=float
)

flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=botm,
    length_units="METERS"
)

strt = np.full(
    (nlay, nrow, ncol),
    initial_head,
    dtype=float
)

flopy.mf6.ModflowGwfic(
    gwf,
    strt=strt
)

# ---------------------------------------------------------------------
# 8. FLOW PROPERTIES
# ---------------------------------------------------------------------

k_array = np.full(
    (nlay, nrow, ncol),
    K_initial,
    dtype=float
)

flopy.mf6.ModflowGwfnpf(
    gwf,
    icelltype=1,  # Convertible/unconfined layer
    k=k_array,
    save_specific_discharge=True,
    save_flows=True
)

# Transient unconfined storage.
# Sy controls unconfined storage changes; Ss is also supplied.
flopy.mf6.ModflowGwfsto(
    gwf,
    iconvert=1,
    ss=Ss_initial,
    sy=Sy_initial,
    transient={kper: True for kper in range(nper)}
)

# ---------------------------------------------------------------------
# 9. RECHARGE
# ---------------------------------------------------------------------
# Uniform provisional recharge for every cell and stress period.
# Replace with a verified recharge series or spatial array in future.

rch_data = {
    kper: np.full(
        (nrow, ncol),
        recharge_m_day,
        dtype=float
    )
    for kper in range(nper)
}

flopy.mf6.ModflowGwfrcha(
    gwf,
    recharge=rch_data,
    pname="RECHARGE"
)

# ---------------------------------------------------------------------
# 10. ILLUSTRATIVE CONSTANT-HEAD BOUNDARIES
# ---------------------------------------------------------------------
# West and east edge heads are assumed for numerical testing.
# They are NOT validated physical boundaries for Alwar.
#
# CHD data are supplied separately for each stress period.

chd_data = {}

for kper in range(nper):
    entries = []

    for i in range(nrow):
        entries.append([(0, i, 0), west_head])
        entries.append([(0, i, ncol - 1), east_head])

    chd_data[kper] = entries

flopy.mf6.ModflowGwfchd(
    gwf,
    stress_period_data=chd_data,
    pname="ILLUSTRATIVE_CHD"
)

# ---------------------------------------------------------------------
# 11. PUMPING
# ---------------------------------------------------------------------
# No pumping wells are assigned.
# Do not invent pumping rates or well locations.
# Add WEL stress-period data after the well coordinates, pumping
# rates, dates, and screened aquifer intervals have been verified.

# ---------------------------------------------------------------------
# 12. OUTPUT CONTROL
# ---------------------------------------------------------------------

flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord="alwar_gwf.hds",
    budget_filerecord="alwar_gwf.cbc",
    saverecord=[
        ("HEAD", "ALL"),
        ("BUDGET", "ALL")
    ],
    printrecord=[
        ("HEAD", "LAST"),
        ("BUDGET", "LAST")
    ]
)

# ---------------------------------------------------------------------
# 13. WRITE MODEL INPUTS AND RUN
# ---------------------------------------------------------------------

print("\nWriting MODFLOW input files...")
sim.write_simulation()

print("\nRunning MODFLOW 6...")
success, report = sim.run_simulation(
    silent=False,
    report=True
)

print("\nSimulation successful:", success)

if report:
    print("\n".join(report[-25:]))

if not success:
    list_file = SIM_WS / "mfsim.lst"

    if list_file.exists():
        print("\nLast lines from mfsim.lst:")
        print("\n".join(
            list_file.read_text(errors="replace").splitlines()[-60:]
        ))

    raise RuntimeError(
        "MODFLOW 6 simulation failed. Review the listing output above."
    )

# ---------------------------------------------------------------------
# 14. READ SIMULATED HEADS
# ---------------------------------------------------------------------

head_path = SIM_WS / "alwar_gwf.hds"
budget_path = SIM_WS / "alwar_gwf.cbc"

if not head_path.exists():
    raise FileNotFoundError(
        f"Simulation completed but head file was not found: {head_path}"
    )

hds = flopy.utils.HeadFile(str(head_path))
times = hds.get_times()

head_final = hds.get_data(totim=times[-1])

# Replace non-finite/extreme values for plotting.
head_final = np.where(
    np.isfinite(head_final) & (np.abs(head_final) < 1.0e20),
    head_final,
    np.nan
)

# Save the final layer as a CSV.
head_df = pd.DataFrame(head_final[0])
head_df.to_csv(
    OUT / "final_head_grid.csv",
    index=False
)

# Save every simulated time's head statistics.
time_summary = []

for t in times:
    h = hds.get_data(totim=t)
    h = np.where(
        np.isfinite(h) & (np.abs(h) < 1.0e20),
        h,
        np.nan
    )

    time_summary.append({
        "time_days": float(t),
        "minimum_head_m": float(np.nanmin(h)),
        "maximum_head_m": float(np.nanmax(h)),
        "mean_head_m": float(np.nanmean(h))
    })

pd.DataFrame(time_summary).to_csv(
    OUT / "head_statistics_by_time.csv",
    index=False
)

# ---------------------------------------------------------------------
# 15. PLOT FINAL HEAD MAP
# ---------------------------------------------------------------------

x = np.arange(ncol) * delr + delr / 2.0
y = np.arange(nrow) * delc + delc / 2.0

fig, ax = plt.subplots(figsize=(9, 7))

mesh = ax.imshow(
    head_final[0],
    origin="upper",
    extent=[0, ncol * delr, nrow * delc, 0],
    aspect="equal"
)

fig.colorbar(
    mesh,
    ax=ax,
    label="Simulated hydraulic head (m; illustrative datum)"
)

if np.isfinite(head_final[0]).sum() > 1:
    ax.contour(
        x,
        y,
        head_final[0],
        levels=8,
        colors="black",
        linewidths=0.6
    )

ax.set_xlabel("Local model X distance (m)")
ax.set_ylabel("Local model Y distance (m)")
ax.set_title("Alwar First-Pass MODFLOW 6 — Final Simulated Heads")

plt.tight_layout()
plt.savefig(
    OUT / "final_head_map.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

# ---------------------------------------------------------------------
# 16. PLOT MEAN HEAD THROUGH TIME
# ---------------------------------------------------------------------

time_df = pd.DataFrame(time_summary)

plt.figure(figsize=(8, 4))
plt.plot(
    time_df["time_days"],
    time_df["mean_head_m"],
    marker="o"
)
plt.xlabel("Simulation time (days)")
plt.ylabel("Mean simulated head (m)")
plt.title("Mean Simulated Head Through Time")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    OUT / "mean_head_timeseries.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

# ---------------------------------------------------------------------
# 17. SENSITIVITY SCENARIO REGISTER
# ---------------------------------------------------------------------
# This records proposed scenarios. They have NOT been run automatically.
# Run and compare these scenarios after confirming the model setup.

scenarios = pd.DataFrame([
    {
        "scenario": "Low_K",
        "K_m_day": K_min,
        "recharge_mm_year": recharge_mm_year,
        "status": "Proposed; not run"
    },
    {
        "scenario": "Initial_K",
        "K_m_day": K_initial,
        "recharge_mm_year": recharge_mm_year,
        "status": "Base prototype"
    },
    {
        "scenario": "High_K",
        "K_m_day": K_max,
        "recharge_mm_year": recharge_mm_year,
        "status": "Proposed; not run"
    },
    {
        "scenario": "Low_recharge",
        "K_m_day": K_initial,
        "recharge_mm_year": 50.0,
        "status": "Proposed; not run"
    },
    {
        "scenario": "High_recharge",
        "K_m_day": K_initial,
        "recharge_mm_year": 200.0,
        "status": "Proposed; not run"
    }
])

scenarios.to_csv(
    OUT / "sensitivity_scenarios_not_run.csv",
    index=False
)

# ---------------------------------------------------------------------
# 18. SAVE RUN SUMMARY
# ---------------------------------------------------------------------

summary = {
    "model": "Alwar first-pass MODFLOW 6",
    "flopy_version": flopy.__version__,
    "modflow_executable": str(MF6_EXE),
    "simulation_success": bool(success),
    "number_of_layers": int(nlay),
    "number_of_rows": int(nrow),
    "number_of_columns": int(ncol),
    "cell_width_m": float(delr),
    "cell_height_m": float(delc),
    "layer_thickness_m": float(layer_thickness),
    "transmissivity_initial_m2_day": float(T_initial),
    "hydraulic_conductivity_initial_m_day": float(K_initial),
    "specific_yield_provisional": float(Sy_initial),
    "specific_storage_provisional_per_m": float(Ss_initial),
    "recharge_provisional_mm_year": float(recharge_mm_year),
    "pumping_assigned": False,
    "calibrated": False,
    "real_district_boundary_used": False,
    "total_simulated_days": float(times[-1]),
    "final_minimum_head_m": float(np.nanmin(head_final)),
    "final_maximum_head_m": float(np.nanmax(head_final)),
    "final_mean_head_m": float(np.nanmean(head_final)),
    "scientific_warning": (
        "Numerical prototype only. Geometry, recharge, boundaries, "
        "and storage parameters are provisional. Not calibrated."
    )
}

with open(OUT / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# ---------------------------------------------------------------------
# 19. PACKAGE ALL OUTPUTS
# ---------------------------------------------------------------------

zip_path = Path("/content/Alwar_MODFLOW6_FirstPass.zip")

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:
    for p in OUT.rglob("*"):
        if p.is_file():
            zf.write(
                p,
                arcname=str(p.relative_to(OUT.parent))
            )

print("\n" + "=" * 70)
print("MODEL RUN FINISHED")
print("=" * 70)
print(json.dumps(summary, indent=2))

print("\nOutput directory:", OUT)
print("ZIP archive:", zip_path)

print("\nFiles created:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(" -", p)

# Download the ZIP directly in Google Colab.
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print(
        "\nNot running inside Google Colab. "
        "Download the ZIP from the output path shown above."
    )


## 3. Calibration preparation

**[orig #33]** Calibration preparation (found 0 numeric water levels — column detection failed)  
<sub>✅ ran OK</sub>

In [ ]:

# =====================================================================
# ALWAR GROUNDWATER CALIBRATION PREPARATION — MODFLOW 6 / FLOPY
# Google Colab
#
# Stage 1: Audit cleaned groundwater observations
# Stage 2: Prepare observed-head and well-to-cell templates
# Stage 3: Calibrate K and recharge only when valid targets are ready
#
# IMPORTANT:
# The existing 20x20 model has an illustrative local grid.
# Geographic coordinates cannot be mapped scientifically to it until
# a real GIS grid / coordinate transformation is supplied.
# =====================================================================

import os
import sys
import json
import shutil
import subprocess
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------- 1. INSTALL / IMPORT ------------------------

try:
    import flopy
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "flopy"
    ])
    import flopy

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("FloPy version:", flopy.__version__)

# ------------------------- 2. PATHS ----------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

GW_FOLDER = (
    DRIVE_ROOT / "gw_framework" / "data" /
    "Rajasthan_gw_data" /
    "Groundwater_Cleaning_According_to_User_Rules"
)

# New output folder. Source data are never edited.
OUT = DRIVE_ROOT / "Alwar_Groundwater_Calibration"
OUT.mkdir(parents=True, exist_ok=True)

CLEANED_LONG = GW_FOLDER / "groundwater_cleaned_long_seasonal.csv"
CLEANED_WIDE = GW_FOLDER / "groundwater_cleaned_wide_source_preserved.csv"

# The existing prototype model may still exist in the current runtime.
MODEL_DIR = Path("/content/Alwar_MODFLOW6_FirstPass/simulation")
MF6_EXE = Path("/content/modflow6_bin/mf6")

print("\nCalibration output:", OUT)
print("Long groundwater file exists:", CLEANED_LONG.exists())
print("Wide groundwater file exists:", CLEANED_WIDE.exists())
print("Existing simulation folder exists:", MODEL_DIR.exists())

# ------------------------- 3. LOCATE SOURCE DATA ----------------------

if CLEANED_LONG.exists():
    source_path = CLEANED_LONG
elif CLEANED_WIDE.exists():
    source_path = CLEANED_WIDE
else:
    raise FileNotFoundError(
        "The expected cleaned groundwater files were not found.\n"
        "Expected folder:\n"
        f"{GW_FOLDER}\n"
        "Check the Google Drive path and filenames."
    )

df = pd.read_csv(source_path, low_memory=False)
print("\nSource file:", source_path)
print("Rows:", len(df))
print("Columns:", list(df.columns))

# Column names are normalized only for matching.
# Original source columns and source files are preserved.
def norm_col(c):
    return (
        str(c).strip().lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )

normalized = {c: norm_col(c) for c in df.columns}

def find_col(candidates):
    # Exact alias match first
    for candidate in candidates:
        for original, normed in normalized.items():
            if normed == candidate:
                return original

    # Then substring match
    for candidate in candidates:
        for original, normed in normalized.items():
            if candidate in normed:
                return original

    return None

well_col = find_col([
    "well_id", "wellid", "well_code", "site_id",
    "unique_id", "station_id", "id"
])

lat_col = find_col(["latitude", "lat"])
lon_col = find_col(["longitude", "long", "lon"])

date_col = find_col([
    "date", "observation_date", "measurement_date",
    "reading_date", "sample_date"
])

season_col = find_col([
    "season", "season_name", "monitoring_season"
])

year_col = find_col(["year", "observation_year"])

status_col = find_col([
    "status", "reading_status", "observation_status",
    "water_level_status"
])

water_level_col = find_col([
    "water_level_m_bgl", "water_level_bgl",
    "depth_to_water", "depth_to_water_m",
    "water_level_m", "water_level", "wl_value",
    "numeric_reading", "reading_value"
])

block_col = find_col(["block", "groundwater_block"])

print("\nDetected column mapping:")
for name, value in {
    "well_id": well_col,
    "latitude": lat_col,
    "longitude": lon_col,
    "date": date_col,
    "season": season_col,
    "year": year_col,
    "status": status_col,
    "water_level": water_level_col,
    "block": block_col
}.items():
    print(f"{name}: {value}")

# Save source-column audit.
pd.DataFrame({
    "source_column": list(df.columns),
    "normalized_for_matching": [normalized[c] for c in df.columns],
    "dtype": [str(df[c].dtype) for c in df.columns]
}).to_csv(OUT / "01_source_column_audit.csv", index=False)

# ------------------------- 4. OBSERVATION STATUS AUDIT ---------------

if status_col:
    status = df[status_col].astype("string").str.strip().str.lower()
else:
    status = pd.Series("unknown", index=df.index, dtype="string")

# Find likely numeric values. Do not convert "Dry" into a number.
if water_level_col:
    numeric_value = pd.to_numeric(
        df[water_level_col].astype("string")
        .str.replace("*", "", regex=False)
        .str.strip(),
        errors="coerce"
    )
else:
    numeric_value = pd.Series(np.nan, index=df.index)

dry_mask = status.eq("dry") | (
    water_level_col is not None and
    df[water_level_col].astype("string").str.strip().str.lower().eq("dry")
)

missing_mask = numeric_value.isna() & ~dry_mask

audit = pd.DataFrame({
    "record_number": np.arange(1, len(df) + 1),
    "source_row_index": df.index,
    "numeric_candidate": numeric_value,
    "status_detected": status,
    "is_dry": dry_mask,
    "is_missing_or_nonnumeric": missing_mask
})

audit.to_csv(OUT / "02_observation_status_audit.csv", index=False)

print("\nObservation audit:")
print("Total rows:", len(df))
print("Numeric candidates:", int(numeric_value.notna().sum()))
print("Dry observations:", int(dry_mask.sum()))
print("Missing/nonnumeric observations:", int(missing_mask.sum()))

# ------------------------- 5. PREPARE OBSERVATION TABLE ---------------

obs = df.copy()

# Preserve source columns; append standardized helper columns.
obs["calibration_numeric_candidate"] = numeric_value
obs["calibration_is_dry"] = dry_mask
obs["calibration_is_missing"] = missing_mask

if well_col:
    obs["calibration_well_id"] = df[well_col].astype("string").str.strip()
else:
    obs["calibration_well_id"] = pd.NA

if lat_col:
    obs["calibration_latitude"] = pd.to_numeric(
        df[lat_col], errors="coerce"
    )
else:
    obs["calibration_latitude"] = np.nan

if lon_col:
    obs["calibration_longitude"] = pd.to_numeric(
        df[lon_col], errors="coerce"
    )
else:
    obs["calibration_longitude"] = np.nan

if date_col:
    obs["calibration_date"] = pd.to_datetime(
        df[date_col], errors="coerce", dayfirst=True
    )
elif year_col:
    # Year alone is not a precise observation date.
    obs["calibration_date"] = pd.to_datetime(
        pd.to_numeric(df[year_col], errors="coerce"),
        format="%Y", errors="coerce"
    )
else:
    obs["calibration_date"] = pd.NaT

if season_col:
    obs["calibration_season"] = df[season_col].astype("string")
else:
    obs["calibration_season"] = pd.NA

if block_col:
    obs["calibration_block"] = df[block_col].astype("string")
else:
    obs["calibration_block"] = pd.NA

# The detected numeric column may represent depth below ground,
# depth to water, or another convention. We do NOT assume it is head.
obs["observed_head_m"] = np.nan
obs["model_row"] = np.nan
obs["model_col"] = np.nan
obs["stress_period"] = np.nan
obs["ground_elevation_m"] = np.nan

# Export every source record with helper columns for inspection.
obs.to_csv(
    OUT / "03_groundwater_observations_with_calibration_fields.csv",
    index=False
)

# Export numeric candidates separately for manual selection.
numeric_obs = obs.loc[obs["calibration_numeric_candidate"].notna()].copy()
numeric_obs.to_csv(
    OUT / "04_numeric_observation_candidates.csv",
    index=False
)

# ------------------------- 6. CALIBRATION TEMPLATE --------------------

# This template must be completed using verified coordinate-to-grid
# mapping and a known vertical datum.
#
# Required columns:
# calibration_well_id, calibration_date, observed_head_m,
# model_row, model_col, stress_period
#
# For depth-to-water readings in m bgl:
# observed_head_m = surveyed_ground_elevation_m - depth_to_water_m
# ONLY if the elevation and water-level reference conventions match.

template_cols = [
    "calibration_well_id",
    "calibration_date",
    "calibration_season",
    "calibration_block",
    "calibration_latitude",
    "calibration_longitude",
    "calibration_numeric_candidate",
    "ground_elevation_m",
    "observed_head_m",
    "model_row",
    "model_col",
    "stress_period"
]

template = obs[template_cols].copy()

template_path = OUT / "05_CALIBRATION_TARGETS_TEMPLATE.csv"
template.to_csv(template_path, index=False)

print("\nCreated calibration target template:")
print(template_path)

# ------------------------- 7. MODEL READINESS CHECK -------------------

if not MODEL_DIR.exists():
    print(
        "\nExisting model files are not present in this runtime. "
        "Upload/extract your downloaded model ZIP before calibration."
    )

required_model_files = [
    "mfsim.nam",
    "alwar_gwf.nam"
]

if MODEL_DIR.exists():
    for fname in required_model_files:
        print(fname, "exists:", (MODEL_DIR / fname).exists())

# ------------------------- 8. CALIBRATION TARGET FILE -----------------
# The script will only calibrate if you fill the template and save a
# separate file named 06_CALIBRATION_TARGETS_READY.csv in OUT.
#
# Do not simply rename the blank template. It needs:
# - real model row and column indices (zero-based)
# - observed hydraulic head in metres
# - stress_period 0 to 11 for the current 12-period model
#
# If observations are only depth-to-water, ground elevations are needed
# to derive hydraulic head. If the model grid is still the illustrative
# 20x20 grid, geographic coordinates cannot be mapped validly yet.

ready_path = OUT / "06_CALIBRATION_TARGETS_READY.csv"

def calibration_readiness_check(path):
    if not path.exists():
        return False, "Ready target file has not been created."

    targets = pd.read_csv(path)

    required = [
        "observed_head_m", "model_row", "model_col", "stress_period"
    ]

    missing_cols = [c for c in required if c not in targets.columns]
    if missing_cols:
        return False, f"Missing required columns: {missing_cols}"

    for c in required:
        targets[c] = pd.to_numeric(targets[c], errors="coerce")

    targets = targets.dropna(subset=required)

    if len(targets) < 3:
        return False, "At least 3 valid targets are required for this check."

    if not targets["model_row"].between(0, 19).all():
        return False, "model_row must be 0–19 for the current prototype."

    if not targets["model_col"].between(0, 19).all():
        return False, "model_col must be 0–19 for the current prototype."

    if not targets["stress_period"].between(0, 11).all():
        return False, "stress_period must be 0–11 for the current model."

    if not np.allclose(
        targets["stress_period"],
        np.round(targets["stress_period"])
    ):
        return False, "stress_period must contain integers."

    if targets["observed_head_m"].nunique() < 2:
        return False, "Observed heads have no meaningful variation."

    return True, targets

ready, result = calibration_readiness_check(ready_path)

# ------------------------- 9. RUN CALIBRATION IF READY ----------------
# Parameters optimized: uniform K and uniform recharge.
# Objective: minimize RMSE between simulated and observed heads.
#
# This is an initial parameter-fitting exercise, not a substitute for
# checking conceptual model validity, boundary conditions, or uncertainty.
#
# Calibration is skipped unless valid targets and existing MODFLOW files
# are available.

if not ready:
    print("\nCALIBRATION NOT RUN.")
    print("Reason:", result)
    print("\nNext steps:")
    print("1. Inspect 04_numeric_observation_candidates.csv.")
    print("2. Verify whether readings are depth-to-water or hydraulic head.")
    print("3. Obtain ground elevations if readings are depth below ground.")
    print("4. Build a real GIS model grid and map each well to a cell.")
    print("5. Fill 05_CALIBRATION_TARGETS_TEMPLATE.csv.")
    print("6. Save the completed file as:")
    print(ready_path)

else:
    if not MODEL_DIR.exists():
        raise FileNotFoundError(
            "Calibration targets are ready but model files are absent. "
            "Restore the model simulation folder first."
        )

    if not MF6_EXE.exists():
        raise FileNotFoundError(
            f"MODFLOW executable not found: {MF6_EXE}"
        )

    targets = result.copy()
    targets["model_row"] = targets["model_row"].astype(int)
    targets["model_col"] = targets["model_col"].astype(int)
    targets["stress_period"] = targets["stress_period"].astype(int)

    # Load existing model.
    sim = flopy.mf6.MFSimulation.load(
        sim_ws=str(MODEL_DIR),
        exe_name=str(MF6_EXE),
        verbosity_level=0
    )

    gwf = sim.get_model("alwar_gwf")
    if gwf is None:
        raise RuntimeError(
            "Model 'alwar_gwf' was not found in the simulation."
        )

    npf = gwf.get_package("npf")
    rcha = gwf.get_package("rcha")

    if npf is None or rcha is None:
        raise RuntimeError(
            "Required NPF or RCHA package is missing."
        )

    # Initial K bounds derived from the CGWB T range divided by the
    # assumed 50 m thickness. These bounds remain provisional.
    k_lo = 27.0 / 50.0
    k_hi = 662.0 / 50.0

    # Recharge search bounds are deliberately broad sensitivity bounds,
    # not measured Alwar recharge rates.
    rch_lo = 0.0
    rch_hi = 0.0008  # m/day, about 292 mm/year

    # Calibrate log10(K) and log10(recharge) when recharge is positive.
    # Zero recharge is handled as a separate candidate.
    from scipy.optimize import differential_evolution

    head_file = MODEL_DIR / "alwar_gwf.hds"

    def run_candidate(log_params):
        K = 10.0 ** float(log_params[0])
        recharge = 10.0 ** float(log_params[1])

        # Update spatially uniform K.
        npf.k.set_data(
            np.full((1, 20, 20), K, dtype=float)
        )

        # Update recharge for all stress periods.
        recharge_data = {
            kper: np.full((20, 20), recharge, dtype=float)
            for kper in range(12)
        }
        rcha.recharge.set_data(recharge_data)

        sim.write_simulation()

        ok, run_report = sim.run_simulation(
            silent=True,
            report=True
        )

        if not ok or not head_file.exists():
            return 1.0e6

        hds = flopy.utils.HeadFile(str(head_file))
        model_times = hds.get_times()

        residuals = []

        for _, row in targets.iterrows():
            sp = int(row["stress_period"])
            if sp >= len(model_times):
                continue

            head = hds.get_data(totim=model_times[sp])
            simulated = head[
                0, int(row["model_row"]), int(row["model_col"])
            ]
            observed = float(row["observed_head_m"])

            if not np.isfinite(simulated) or abs(simulated) > 1.0e20:
                continue

            residuals.append(simulated - observed)

        if len(residuals) < 3:
            return 1.0e6

        return float(np.sqrt(np.mean(np.square(residuals))))

    bounds = [
        (np.log10(k_lo), np.log10(k_hi)),
        (np.log10(1.0e-7), np.log10(rch_hi))
    ]

    print("\nStarting calibration search. This may take time.")

    result_opt = differential_evolution(
        run_candidate,
        bounds=bounds,
        seed=42,
        maxiter=12,
        popsize=6,
        polish=False,
        workers=1,
        updating="immediate"
    )

    best_K = 10.0 ** result_opt.x[0]
    best_recharge = 10.0 ** result_opt.x[1]
    best_rmse = float(result_opt.fun)

    # Run final best parameter combination and save final heads.
    final_rmse = run_candidate(result_opt.x)

    hds = flopy.utils.HeadFile(str(head_file))
    model_times = hds.get_times()

    residual_rows = []
    for _, row in targets.iterrows():
        sp = int(row["stress_period"])
        if sp >= len(model_times):
            continue

        head = hds.get_data(totim=model_times[sp])
        simulated = float(
            head[0, int(row["model_row"]), int(row["model_col"])]
        )
        observed = float(row["observed_head_m"])

        residual_rows.append({
            "well_id": row.get("calibration_well_id", ""),
            "stress_period": sp,
            "model_row": int(row["model_row"]),
            "model_col": int(row["model_col"]),
            "observed_head_m": observed,
            "simulated_head_m": simulated,
            "residual_sim_minus_obs_m": simulated - observed
        })

    residual_df = pd.DataFrame(residual_rows)
    residual_df.to_csv(
        OUT / "07_calibration_residuals.csv",
        index=False
    )

    calibration_summary = {
        "calibration_method": "Differential evolution; uniform K and recharge",
        "successful_optimizer_run": bool(result_opt.success),
        "optimizer_message": str(result_opt.message),
        "number_of_targets": int(len(residual_df)),
        "best_K_m_day": float(best_K),
        "best_recharge_m_day": float(best_recharge),
        "best_recharge_mm_year": float(best_recharge * 365.25 * 1000),
        "RMSE_m": float(best_rmse),
        "interpretation": (
            "Conditional initial fit only. Results are not defensible as "
            "district calibration unless the model grid, observation heads, "
            "well-to-cell mapping, boundaries and conceptual model are valid."
        )
    }

    with open(OUT / "08_calibration_summary.json", "w") as f:
        json.dump(calibration_summary, f, indent=2)

    print("\nCalibration result:")
    print(json.dumps(calibration_summary, indent=2))

    # Plot residuals
    if not residual_df.empty:
        plt.figure(figsize=(8, 4))
        plt.scatter(
            residual_df["observed_head_m"],
            residual_df["simulated_head_m"]
        )
        minv = min(
            residual_df["observed_head_m"].min(),
            residual_df["simulated_head_m"].min()
        )
        maxv = max(
            residual_df["observed_head_m"].max(),
            residual_df["simulated_head_m"].max()
        )
        plt.plot([minv, maxv], [minv, maxv], linestyle="--")
        plt.xlabel("Observed hydraulic head (m)")
        plt.ylabel("Simulated hydraulic head (m)")
        plt.title("Calibration: Observed vs Simulated Heads")
        plt.tight_layout()
        plt.savefig(
            OUT / "09_observed_vs_simulated.png",
            dpi=200,
            bbox_inches="tight"
        )
        plt.show()

# ------------------------- 10. ZIP REPORTS ---------------------------

zip_path = Path("/content/Alwar_Groundwater_Calibration_Reports.zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for p in OUT.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(OUT.parent)))

print("\nFinished.")
print("Calibration workspace:", OUT)
print("Report archive:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Download the archive from the path shown above.")


## 4. Save to Drive

**[orig #34]** Copy the model ZIP from `/content` to Drive `gw_framework/outputs`  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import shutil

from google.colab import drive
drive.mount("/content/drive")

# Existing ZIP created by your MODFLOW script
source_zip = Path("/content/Alwar_MODFLOW6_FirstPass.zip")

# Destination in Google Drive
drive_folder = Path(
    "/content/drive/MyDrive/gw_framework/outputs"
)
drive_folder.mkdir(parents=True, exist_ok=True)

destination_zip = drive_folder / "Alwar_MODFLOW6_FirstPass.zip"

if source_zip.exists():
    shutil.copy2(source_zip, destination_zip)
    print("SUCCESS: ZIP saved to Google Drive")
    print("Location:", destination_zip)
    print("Size (MB):", round(destination_zip.stat().st_size / (1024**2), 2))
else:
    print("ZIP not found in the current Colab session:", source_zip)
    print(
        "If the runtime restarted, upload your downloaded "
        "Alwar_MODFLOW6_FirstPass.zip to Colab, then run this cell again."
    )
